In [2]:
# ============================================================
# 12_diebold_mariano_final_test.ipynb
#
# Diebold-Mariano tests for final 2024-latest evaluation

# Main output:
#   dm_winner_vs_rest_primary_scaled_abs_clean.csv
#
# Main loss:
#   scaled_abs_error = |y - pred| / MASE_scale
#
# Main hypothesis in winner-vs-rest:
#   H0: equal predictive accuracy
#   H1: winner has lower expected loss than competitor
#
# Holm-Bonferroni corrections included:
#   - block m=6: scope × disease × horizon
#   - scope_disease m=24: scope × disease
#   - scope m=48: scope
#   - all m=96: whole winner-vs-rest table
#
# No two-sided columns are kept in the final clean outputs.
# ============================================================

import sys
from pathlib import Path
from itertools import combinations
from math import erfc, sqrt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 0. Locate folders
# ============================================================

try:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import get_project_paths
    RESULTS_DIR = get_project_paths(PROJECT_ROOT).results
except Exception:
    RESULTS_DIR = Path.cwd().parent / "results"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results folder not found: {RESULTS_DIR}")

FINAL_TEST_DIR = RESULTS_DIR / "final_test_2024_2025"

if not FINAL_TEST_DIR.exists():
    candidates = list(RESULTS_DIR.rglob("finaltest_all_predictions_long.csv"))
    if not candidates:
        raise FileNotFoundError("Could not find finaltest_all_predictions_long.csv.")
    FINAL_TEST_DIR = candidates[0].parent

OUT_DIR = FINAL_TEST_DIR / "diebold_mariano_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using final-test folder:", FINAL_TEST_DIR)
print("Saving outputs to:", OUT_DIR)


# ============================================================
# 1. Load predictions and MASE scales
# ============================================================

pred_path = FINAL_TEST_DIR / "finaltest_all_predictions_long.csv"
scale_path = FINAL_TEST_DIR / "finaltest_mase_scales_2014_2023.csv"

if not pred_path.exists():
    raise FileNotFoundError(f"Missing: {pred_path}")

if not scale_path.exists():
    raise FileNotFoundError(f"Missing: {scale_path}")

pred_long = pd.read_csv(pred_path)
scales = pd.read_csv(scale_path)

pred_long.columns = [c.strip() for c in pred_long.columns]
scales.columns = [c.strip() for c in scales.columns]

required_pred_cols = ["origin", "target", "disease", "location", "h", "y", "pred", "model"]
required_scale_cols = ["disease", "location", "mase_scale"]

missing_pred = [c for c in required_pred_cols if c not in pred_long.columns]
missing_scale = [c for c in required_scale_cols if c not in scales.columns]

if missing_pred:
    raise ValueError(f"Missing columns in predictions table: {missing_pred}")

if missing_scale:
    raise ValueError(f"Missing columns in scale table: {missing_scale}")

pred_long["origin"] = pd.to_datetime(pred_long["origin"])
pred_long["target"] = pd.to_datetime(pred_long["target"])
pred_long["disease"] = pred_long["disease"].astype(str)
pred_long["location"] = pred_long["location"].astype(str)
pred_long["model"] = pred_long["model"].astype(str)
pred_long["h"] = pd.to_numeric(pred_long["h"], errors="coerce").astype(int)
pred_long["y"] = pd.to_numeric(pred_long["y"], errors="coerce")
pred_long["pred"] = pd.to_numeric(pred_long["pred"], errors="coerce")

scales["disease"] = scales["disease"].astype(str)
scales["location"] = scales["location"].astype(str)
scales["mase_scale"] = pd.to_numeric(scales["mase_scale"], errors="coerce")

pred_long = pred_long.merge(
    scales[["disease", "location", "mase_scale"]],
    on=["disease", "location"],
    how="left"
)

if pred_long["mase_scale"].isna().any():
    bad = pred_long[pred_long["mase_scale"].isna()][["disease", "location"]].drop_duplicates()
    raise ValueError("Missing mase_scale for some disease-location pairs:\n" + str(bad))

pred_long = pred_long.dropna(subset=["y", "pred", "mase_scale"]).copy()

print("Usable prediction rows:", pred_long.shape)
print("Models:")
for m in sorted(pred_long["model"].unique()):
    print("-", m)

coverage = (
    pred_long
    .groupby(["disease", "h", "model"], as_index=False)
    .agg(
        n_points=("pred", "size"),
        n_countries=("location", "nunique")
    )
    .sort_values(["disease", "h", "model"])
)

print("\nCoverage check:")
display(coverage)


# ============================================================
# 2. Loss columns
# ============================================================

pred_long["error"] = pred_long["y"] - pred_long["pred"]
pred_long["abs_error"] = pred_long["error"].abs()
pred_long["scaled_abs_error"] = pred_long["abs_error"] / pred_long["mase_scale"]

LOSS_COL = "scaled_abs_error"

COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]


# ============================================================
# 3. Recompute summaries and winners
# ============================================================

def compute_country_h_from_long(df):
    rows = []

    for (disease, location, h, model), sub in df.groupby(["disease", "location", "h", "model"]):
        sub = sub.dropna(subset=["y", "pred", "mase_scale"]).copy()

        if len(sub) == 0:
            continue

        err = sub["y"].values - sub["pred"].values
        mae = float(np.mean(np.abs(err)))
        rmse = float(np.sqrt(np.mean(err ** 2)))
        scale = float(sub["mase_scale"].iloc[0])
        mase = mae / scale if np.isfinite(scale) and scale > 0 else np.nan

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "model": model,
            "MAE": mae,
            "RMSE": rmse,
            "MASE": mase,
            "n": int(len(sub))
        })

    return pd.DataFrame(rows)


def compute_horizon_summary(df):
    country_h = compute_country_h_from_long(df)

    horizon_summary = (
        country_h
        .groupby(["disease", "h", "model"], as_index=False)
        .agg(
            MAE=("MAE", "mean"),
            RMSE=("RMSE", "mean"),
            MASE=("MASE", "mean"),
            n_countries=("location", "nunique"),
            n_points=("n", "sum")
        )
        .sort_values(["disease", "h", "MASE", "MAE"])
        .reset_index(drop=True)
    )

    winners = (
        horizon_summary
        .sort_values(["disease", "h", "MASE", "MAE"])
        .groupby(["disease", "h"], group_keys=False)
        .head(1)
        .reset_index(drop=True)
    )

    return country_h, horizon_summary, winners


country_h_global, horizon_summary_global, winners_global = compute_horizon_summary(pred_long)

pred_common6 = pred_long[pred_long["location"].isin(COMMON6)].copy()
country_h_common6, horizon_summary_common6, winners_common6 = compute_horizon_summary(pred_common6)

print("\nGlobal winners:")
display(winners_global)

print("\nCommon6 winners:")
display(winners_common6)


# ============================================================
# 4. DM implementation
# ============================================================

def normal_cdf(x):
    return 0.5 * erfc(-x / sqrt(2.0))


def newey_west_lrv(x, max_lag):
    """
    Long-run variance with Bartlett/Newey-West weights.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    T = len(x)

    if T <= 1:
        return np.nan

    xc = x - np.mean(x)

    gamma0 = np.sum(xc * xc) / T
    lrv = gamma0

    max_lag = int(max_lag)

    for lag in range(1, max_lag + 1):
        if lag >= T:
            break

        gamma = np.sum(xc[lag:] * xc[:-lag]) / T
        weight = 1.0 - lag / (max_lag + 1.0)
        lrv += 2.0 * weight * gamma

    return float(lrv)


def dm_test_loss_diff(diff_series, h, harvey_correction=True):
    """
    Diebold-Mariano test on:
        d_t = loss_A,t - loss_B,t

    In winner-vs-rest:
        A = winner
        B = competitor

    One-sided alternative:
        A better than B  <=>  E[d_t] < 0
    """

    d = pd.Series(diff_series).dropna().astype(float).values
    d = d[np.isfinite(d)]

    T = len(d)
    hac_lag = max(int(h) - 1, 0)

    if T < max(8, int(h) + 2):
        return {
            "dm_stat": np.nan,
            "p_one_sided_A_better": np.nan,
            "mean_loss_diff_A_minus_B": np.nan,
            "n_time_points": T,
            "hac_lag": hac_lag,
            "note": "too_few_time_points"
        }

    mean_d = float(np.mean(d))
    lrv = newey_west_lrv(d, hac_lag)

    if not np.isfinite(lrv) or lrv < 1e-14:
        if abs(mean_d) < 1e-14:
            return {
                "dm_stat": 0.0,
                "p_one_sided_A_better": 0.5,
                "mean_loss_diff_A_minus_B": mean_d,
                "n_time_points": T,
                "hac_lag": hac_lag,
                "note": "zero_variance_zero_mean"
            }
        else:
            dm_stat = -np.inf if mean_d < 0 else np.inf
            p = 0.0 if mean_d < 0 else 1.0

            return {
                "dm_stat": dm_stat,
                "p_one_sided_A_better": p,
                "mean_loss_diff_A_minus_B": mean_d,
                "n_time_points": T,
                "hac_lag": hac_lag,
                "note": "zero_variance_nonzero_mean"
            }

    dm_stat = mean_d / np.sqrt(lrv / T)

    # Harvey-Leybourne-Newbold finite-sample correction
    if harvey_correction:
        correction_inside = (T + 1 - 2 * h + (h * (h - 1) / T)) / T

        if correction_inside > 0:
            dm_stat = dm_stat * np.sqrt(correction_inside)

    # Left-tail p-value: A has lower loss than B
    p_one_sided_A_better = normal_cdf(dm_stat)

    return {
        "dm_stat": float(dm_stat),
        "p_one_sided_A_better": float(p_one_sided_A_better),
        "mean_loss_diff_A_minus_B": mean_d,
        "n_time_points": int(T),
        "hac_lag": int(hac_lag),
        "note": "ok"
    }


def holm_adjusted_pvalues(pvalues):
    """
    Holm-Bonferroni adjusted p-values.

    Equivalent to the sequential rule:
        compare p_(k) with alpha / (m - k + 1)

    Adjusted p-values are:
        p_adj_(k) = max_{j<=k} [(m-j+1) p_(j)]
    truncated at 1.
    """
    p = np.asarray(pvalues, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)

    valid = np.isfinite(p)
    p_valid = p[valid]

    if len(p_valid) == 0:
        return out

    order = np.argsort(p_valid)
    sorted_p = p_valid[order]
    m = len(sorted_p)

    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0

    for i, pv in enumerate(sorted_p):
        factor = m - i
        adj = factor * pv
        running_max = max(running_max, adj)
        adjusted_sorted[i] = min(running_max, 1.0)

    adjusted_valid = np.empty(m, dtype=float)
    adjusted_valid[order] = adjusted_sorted

    out[valid] = adjusted_valid

    return out


def add_holm_column(df, p_col, group_cols, out_col, m_col):
    df = df.copy()
    df[out_col] = np.nan
    df[m_col] = np.nan

    for _, idx in df.groupby(group_cols).groups.items():
        idx = list(idx)
        pvals = df.loc[idx, p_col].values
        df.loc[idx, out_col] = holm_adjusted_pvalues(pvals)
        df.loc[idx, m_col] = int(np.isfinite(pvals).sum())

    df[m_col] = df[m_col].astype("Int64")

    return df


def add_global_holm_column(df, p_col, out_col, m_col):
    df = df.copy()
    pvals = df[p_col].values
    df[out_col] = holm_adjusted_pvalues(pvals)
    df[m_col] = int(np.isfinite(pvals).sum())
    return df


# ============================================================
# 5. Pairwise DM utilities
# ============================================================

def dm_pair(
    df,
    disease,
    h,
    model_A,
    model_B,
    scope,
    loss_col=LOSS_COL
):
    """
    Model A is tested against model B.
    In winner-vs-rest, A is the winner.

    Data are paired by:
        disease, location, origin, target, h

    Then the loss differential is averaged by target week,
    and DM is applied to the weekly time series.
    """

    sub = df[
        (df["disease"] == disease) &
        (df["h"] == h) &
        (df["model"].isin([model_A, model_B]))
    ].copy()

    if len(sub) == 0:
        return None

    index_cols = ["disease", "location", "origin", "target", "h"]

    wide = (
        sub
        .pivot_table(
            index=index_cols,
            columns="model",
            values=loss_col,
            aggfunc="first"
        )
        .reset_index()
    )

    if model_A not in wide.columns or model_B not in wide.columns:
        return None

    paired = wide.dropna(subset=[model_A, model_B]).copy()

    if len(paired) == 0:
        return None

    paired["loss_diff_A_minus_B"] = paired[model_A] - paired[model_B]

    weekly_diff = (
        paired
        .groupby("target", as_index=True)["loss_diff_A_minus_B"]
        .mean()
        .sort_index()
    )

    dm = dm_test_loss_diff(weekly_diff, h=h, harvey_correction=True)

    mean_A = float(paired[model_A].mean())
    mean_B = float(paired[model_B].mean())

    return {
        "scope": scope,
        "loss": loss_col,
        "disease": disease,
        "h": int(h),
        "model_A": model_A,
        "model_B": model_B,
        "mean_loss_A_panel": mean_A,
        "mean_loss_B_panel": mean_B,
        "mean_loss_diff_A_minus_B": dm["mean_loss_diff_A_minus_B"],
        "dm_stat": dm["dm_stat"],
        "p_one_sided_A_better": dm["p_one_sided_A_better"],
        "n_paired_panel_rows": int(len(paired)),
        "n_time_points": int(dm["n_time_points"]),
        "n_locations": int(paired["location"].nunique()),
        "hac_lag": int(dm["hac_lag"]),
        "note": dm["note"]
    }


# ============================================================
# 6. Winner-vs-rest DM
# ============================================================

def run_winner_vs_rest(df, winners_table, scope):
    rows = []

    for _, row in winners_table.iterrows():
        disease = row["disease"]
        h = int(row["h"])
        winner = row["model"]

        sub = df[
            (df["disease"] == disease) &
            (df["h"] == h)
        ].copy()

        competitors = sorted([m for m in sub["model"].unique() if m != winner])

        for comp in competitors:
            res = dm_pair(
                df=df,
                disease=disease,
                h=h,
                model_A=winner,
                model_B=comp,
                scope=scope,
                loss_col=LOSS_COL
            )

            if res is None:
                continue

            res["winner_model"] = winner
            res["competitor_model"] = comp
            res["winner_mase_from_summary"] = float(row["MASE"])
            rows.append(res)

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    # Rename for clarity
    out = out.rename(columns={
        "model_A": "winner_as_model_A",
        "model_B": "competitor_as_model_B",
        "mean_loss_A_panel": "mean_loss_winner_panel",
        "mean_loss_B_panel": "mean_loss_competitor_panel",
        "mean_loss_diff_A_minus_B": "mean_loss_diff_winner_minus_competitor",
        "p_one_sided_A_better": "p_one_sided_winner_better"
    })

    # one family per scope × disease × horizon
    
    out = add_holm_column(
        out,
        p_col="p_one_sided_winner_better",
        group_cols=["scope", "disease", "h"],
        out_col="p_holm_block_scope_disease_h",
        m_col="m_block_scope_disease_h"
    )

    # Alternative more conservative corrections
    out = add_holm_column(
        out,
        p_col="p_one_sided_winner_better",
        group_cols=["scope", "disease"],
        out_col="p_holm_scope_disease",
        m_col="m_scope_disease"
    )

    out = add_holm_column(
        out,
        p_col="p_one_sided_winner_better",
        group_cols=["scope"],
        out_col="p_holm_scope",
        m_col="m_scope"
    )

    out = add_global_holm_column(
        out,
        p_col="p_one_sided_winner_better",
        out_col="p_holm_all_tests",
        m_col="m_all_tests"
    )

    # Significance flags
    out["sig_unadjusted_5pct"] = out["p_one_sided_winner_better"] < 0.05
    out["sig_holm_block_5pct"] = out["p_holm_block_scope_disease_h"] < 0.05
    out["sig_holm_scope_disease_5pct"] = out["p_holm_scope_disease"] < 0.05
    out["sig_holm_scope_5pct"] = out["p_holm_scope"] < 0.05
    out["sig_holm_all_5pct"] = out["p_holm_all_tests"] < 0.05

    # Sorting 
    # inside each comparison block, sort by competitor's mean loss.
    out = (
        out
        .sort_values(
            ["scope", "disease", "h", "mean_loss_competitor_panel", "dm_stat"],
            ascending=[True, True, True, True, True]
        )
        .reset_index(drop=True)
    )

    out["competitor_rank_by_mean_loss_within_block"] = (
        out
        .groupby(["scope", "disease", "h"])["mean_loss_competitor_panel"]
        .rank(method="first", ascending=True)
        .astype(int)
    )

    # Final column order
    ordered_cols = [
        "scope",
        "loss",
        "disease",
        "h",
        "winner_model",
        "competitor_model",
        "competitor_rank_by_mean_loss_within_block",
        "winner_mase_from_summary",
        "mean_loss_winner_panel",
        "mean_loss_competitor_panel",
        "mean_loss_diff_winner_minus_competitor",
        "dm_stat",
        "p_one_sided_winner_better",
        "p_holm_block_scope_disease_h",
        "m_block_scope_disease_h",
        "p_holm_scope_disease",
        "m_scope_disease",
        "p_holm_scope",
        "m_scope",
        "p_holm_all_tests",
        "m_all_tests",
        "sig_unadjusted_5pct",
        "sig_holm_block_5pct",
        "sig_holm_scope_disease_5pct",
        "sig_holm_scope_5pct",
        "sig_holm_all_5pct",
        "n_paired_panel_rows",
        "n_time_points",
        "n_locations",
        "hac_lag",
        "note"
    ]

    existing = [c for c in ordered_cols if c in out.columns]
    remaining = [c for c in out.columns if c not in existing]

    return out[existing + remaining]


winner_vs_rest_global = run_winner_vs_rest(
    df=pred_long,
    winners_table=winners_global,
    scope="global"
)

winner_vs_rest_common6 = run_winner_vs_rest(
    df=pred_common6,
    winners_table=winners_common6,
    scope="common6"
)

dm_winner_vs_rest = pd.concat(
    [winner_vs_rest_global, winner_vs_rest_common6],
    ignore_index=True
)


# ============================================================
# 7.  all-pairs table, one-sided and oriented
# ============================================================

def run_all_pairs_oriented(df, horizon_summary, scope):
    """
    Runs all model pairs, but orients each pair as:
        model_A = lower mean MASE model in the summary
        model_B = higher mean MASE model

    Therefore p_one_sided_A_better asks whether the better-ranked
    model is significantly better than the worse-ranked model.
    """

    rows = []

    for disease in sorted(df["disease"].unique()):
        for h in sorted(df["h"].unique()):
            sub_summary = horizon_summary[
                (horizon_summary["disease"] == disease) &
                (horizon_summary["h"] == h)
            ].copy()

            models_ordered = (
                sub_summary
                .sort_values(["MASE", "MAE"])["model"]
                .tolist()
            )

            model_rank = {m: i + 1 for i, m in enumerate(models_ordered)}

            for model_1, model_2 in combinations(models_ordered, 2):
                # model_1 is better-ranked by construction
                res = dm_pair(
                    df=df,
                    disease=disease,
                    h=h,
                    model_A=model_1,
                    model_B=model_2,
                    scope=scope,
                    loss_col=LOSS_COL
                )

                if res is None:
                    continue

                res["model_A_rank_by_MASE"] = model_rank[model_1]
                res["model_B_rank_by_MASE"] = model_rank[model_2]
                rows.append(res)

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    out = out.rename(columns={
        "model_A": "better_ranked_model_A",
        "model_B": "worse_ranked_model_B",
        "mean_loss_A_panel": "mean_loss_model_A_panel",
        "mean_loss_B_panel": "mean_loss_model_B_panel",
        "mean_loss_diff_A_minus_B": "mean_loss_diff_A_minus_B",
        "p_one_sided_A_better": "p_one_sided_A_better"
    })

    out = add_holm_column(
        out,
        p_col="p_one_sided_A_better",
        group_cols=["scope", "disease", "h"],
        out_col="p_holm_block_scope_disease_h",
        m_col="m_block_scope_disease_h"
    )

    out = add_holm_column(
        out,
        p_col="p_one_sided_A_better",
        group_cols=["scope"],
        out_col="p_holm_scope",
        m_col="m_scope"
    )

    out = add_global_holm_column(
        out,
        p_col="p_one_sided_A_better",
        out_col="p_holm_all_tests",
        m_col="m_all_tests"
    )

    out["sig_holm_block_5pct"] = out["p_holm_block_scope_disease_h"] < 0.05
    out["sig_holm_scope_5pct"] = out["p_holm_scope"] < 0.05
    out["sig_holm_all_5pct"] = out["p_holm_all_tests"] < 0.05

    out = (
        out
        .sort_values(
            ["scope", "disease", "h", "model_A_rank_by_MASE", "model_B_rank_by_MASE"]
        )
        .reset_index(drop=True)
    )

    return out


dm_all_pairs_global = run_all_pairs_oriented(
    df=pred_long,
    horizon_summary=horizon_summary_global,
    scope="global"
)

dm_all_pairs_common6 = run_all_pairs_oriented(
    df=pred_common6,
    horizon_summary=horizon_summary_common6,
    scope="common6"
)

dm_all_pairs = pd.concat(
    [dm_all_pairs_global, dm_all_pairs_common6],
    ignore_index=True
)


# ============================================================
# 8. Display results
# ============================================================

print("\n============================================================")
print("Winner vs rest — clean one-sided DM table")
print("============================================================")
display(dm_winner_vs_rest)

print("\n============================================================")
print("Significant winner-vs-rest at 5% using main block Holm correction")
print("============================================================")
display(
    dm_winner_vs_rest[
        dm_winner_vs_rest["sig_holm_block_5pct"]
    ].reset_index(drop=True)
)

print("\n============================================================")
print("Adjustment sizes used")
print("============================================================")
adjustment_sizes = (
    dm_winner_vs_rest
    .groupby(["scope", "disease", "h"], as_index=False)
    .agg(
        m_block=("m_block_scope_disease_h", "first"),
        n_rows=("p_one_sided_winner_better", "size")
    )
)
display(adjustment_sizes)


# ============================================================
# 9. Save outputs
# ============================================================

country_h_global.to_csv(OUT_DIR / "dm_country_h_global.csv", index=False)
horizon_summary_global.to_csv(OUT_DIR / "dm_horizon_summary_global.csv", index=False)
winners_global.to_csv(OUT_DIR / "dm_winners_global.csv", index=False)

country_h_common6.to_csv(OUT_DIR / "dm_country_h_common6.csv", index=False)
horizon_summary_common6.to_csv(OUT_DIR / "dm_horizon_summary_common6.csv", index=False)
winners_common6.to_csv(OUT_DIR / "dm_winners_common6.csv", index=False)

dm_winner_vs_rest.to_csv(
    OUT_DIR / "dm_winner_vs_rest_primary_scaled_abs_clean.csv",
    index=False
)

dm_all_pairs.to_csv(
    OUT_DIR / "dm_all_pairs_primary_scaled_abs_clean.csv",
    index=False
)

adjustment_sizes.to_csv(
    OUT_DIR / "dm_adjustment_sizes_used.csv",
    index=False
)

methodology_text = """
Diebold-Mariano methodology

Primary loss:
scaled_abs_error = |y - prediction| / MASE_scale.

For winner-vs-rest comparisons:
model A is always the winner according to mean MASE in the corresponding
scope/disease/horizon table. The one-sided alternative is that the winner
has lower expected loss than the competitor.

Loss differentials:
d_t = loss_winner,t - loss_competitor,t.

Because the data are a panel by country, the row-level loss differentials
are first paired by disease, location, origin, target and horizon. Then they
are averaged by target week. The Diebold-Mariano test is applied to the
weekly series of mean loss differentials.

Variance:
Newey-West/Bartlett long-run variance with lag h-1.

Small sample correction:
Harvey-Leybourne-Newbold correction.

Multiple testing:
The main Holm-Bonferroni correction is applied within each
scope x disease x horizon block. In winner-vs-rest this gives m=6 tests,
because the winner is compared against the other six procedures.

Additional conservative corrections are also reported:
- p_holm_scope_disease: within each scope x disease
- p_holm_scope: within each scope
- p_holm_all_tests: across the whole winner-vs-rest table
"""

with open(OUT_DIR / "dm_methodology_notes.txt", "w", encoding="utf-8") as f:
    f.write(methodology_text)

print("\nSaved outputs to:")
print(OUT_DIR)

print("\nFiles generated:")
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)

Using final-test folder: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025
Saving outputs to: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\diebold_mariano_clean
Usable prediction rows: (49252, 15)
Models:
- DL_GlobalGRU_all_infections_all_countries
- RandomForest(lags=4)
- SARIMA(1,0,0)x(1,0,0)[52] (initial)
- autoARIMA
- ensemble_mean_5
- ensemble_median_5
- rf_global_all_infections_all_countries

Coverage check:


,disease,h,model,n_points,n_countries
0,ARI,1,DL_GlobalGRU_all_infections_all_countries,791,8
1,ARI,1,RandomForest(lags=4),791,8
2,ARI,1,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",791,8
3,ARI,1,autoARIMA,791,8
4,ARI,1,ensemble_mean_5,791,8
5,ARI,1,ensemble_median_5,791,8
6,ARI,1,rf_global_all_infections_all_countries,791,8
7,ARI,2,DL_GlobalGRU_all_infections_all_countries,783,8
8,ARI,2,RandomForest(lags=4),783,8
9,ARI,2,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",783,8



Global winners:


,disease,h,model,MAE,RMSE,MASE,n_countries,n_points
0,ARI,1,ensemble_mean_5,92.464206,133.297746,0.422145,8,791
1,ARI,2,DL_GlobalGRU_all_infections_all_countries,117.652291,165.766120,0.541678,8,783
2,ARI,3,DL_GlobalGRU_all_infections_all_countries,129.736995,183.118465,0.596824,8,775
3,ARI,4,DL_GlobalGRU_all_infections_all_countries,136.774194,193.019065,0.631291,8,767
4,ILI,1,ensemble_mean_5,24.842302,41.712327,0.533985,10,995
5,ILI,2,ensemble_mean_5,32.347626,55.127552,0.732653,10,985
6,ILI,3,ensemble_mean_5,39.210679,66.662411,0.920093,10,975
7,ILI,4,rf_global_all_infections_all_countries,46.523095,76.049622,1.016970,10,965



Common6 winners:


...


Significant winner-vs-rest at 5% using main block Holm correction


,scope,loss,disease,h,winner_model,competitor_model,competitor_rank_by_mean_loss_within_block,winner_mase_from_summary,mean_loss_winner_panel,mean_loss_competitor_panel,...,sig_holm_scope_disease_5pct,sig_holm_scope_5pct,sig_holm_all_5pct,n_paired_panel_rows,n_time_points,n_locations,hac_lag,note,winner_as_model_A,competitor_as_model_B
0,global,scaled_abs_error,ARI,1,ensemble_mean_5,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",3,0.422145,0.422484,0.449686,...,False,False,False,791,101,8,0,ok,ensemble_mean_5,"SARIMA(1,0,0)x(1,0,0)[52] (initial)"
1,global,scaled_abs_error,ARI,1,ensemble_mean_5,rf_global_all_infections_all_countries,4,0.422145,0.422484,0.464655,...,True,True,True,791,101,8,0,ok,ensemble_mean_5,rf_global_all_infections_all_countries
2,global,scaled_abs_error,ARI,1,ensemble_mean_5,autoARIMA,5,0.422145,0.422484,0.465008,...,True,True,True,791,101,8,0,ok,ensemble_mean_5,autoARIMA
3,global,scaled_abs_error,ARI,1,ensemble_mean_5,RandomForest(lags=4),6,0.422145,0.422484,0.537236,...,True,True,True,791,101,8,0,ok,ensemble_mean_5,RandomForest(lags=4)
4,global,scaled_abs_error,ARI,2,DL_GlobalGRU_all_infections_all_countries,RandomForest(lags=4),6,0.541678,0.544023,0.766893,...,True,True,True,783,100,8,1,ok,DL_GlobalGRU_all_infections_all_countries,RandomForest(lags=4)
5,global,scaled_abs_error,ARI,3,DL_GlobalGRU_all_infections_all_countries,RandomForest(lags=4),6,0.596824,0.599525,0.933705,...,True,True,True,775,99,8,2,ok,DL_GlobalGRU_all_infections_all_countries,RandomForest(lags=4)
6,global,scaled_abs_error,ARI,4,DL_GlobalGRU_all_infections_all_countries,autoARIMA,5,0.631291,0.633795,0.822602,...,False,False,False,767,98,8,3,ok,DL_GlobalGRU_all_infections_all_countries,autoARIMA
7,global,scaled_abs_error,ARI,4,DL_GlobalGRU_all_infections_all_countries,RandomForest(lags=4),6,0.631291,0.633795,1.053384,...,True,True,True,767,98,8,3,ok,DL_GlobalGRU_all_infections_all_countries,RandomForest(lags=4)
8,global,scaled_abs_error,ILI,1,ensemble_mean_5,autoARIMA,4,0.533985,0.532664,0.590133,...,True,True,True,995,101,10,0,ok,ensemble_mean_5,autoARIMA
9,global,scaled_abs_error,ILI,1,ensemble_mean_5,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",5,0.533985,0.532664,0.612559,...,True,True,True,995,101,10,0,ok,ensemble_mean_5,"SARIMA(1,0,0)x(1,0,0)[52] (initial)"



Adjustment sizes used


,scope,disease,h,m_block,n_rows
0,common6,ARI,1,6,6
1,common6,ARI,2,6,6
2,common6,ARI,3,6,6
3,common6,ARI,4,6,6
4,common6,ILI,1,6,6
5,common6,ILI,2,6,6
6,common6,ILI,3,6,6
7,common6,ILI,4,6,6
8,global,ARI,1,6,6
9,global,ARI,2,6,6



Saved outputs to:
C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\diebold_mariano_clean

Files generated:
- dm_adjustment_sizes_used.csv
- dm_all_pairs_primary_scaled_abs_clean.csv
- dm_country_h_common6.csv
- dm_country_h_global.csv
- dm_horizon_summary_common6.csv
- dm_horizon_summary_global.csv
- dm_methodology_notes.txt
- dm_winner_vs_rest_primary_scaled_abs_clean.csv
- dm_winners_common6.csv
- dm_winners_global.csv
